In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pygraphon.graphons.StepGraphon import StepGraphon
import numpy as np
from pygraphon.plots.plot_matrices import spy
import matplotlib.pyplot as plt
from pygraphon.utils.utils_graph import edge_density
from tqdm import tqdm
from pygraphon.plots.plots_graphon import plot_sample

from SmoothNetHist import SmoothNetHist
from experiment_smoothing import smooth_hist, initialisation_hist, smoothing_histLC, flat_to_tensor

from gen_sbm import AssociativeFullyRandom

In [ ]:
#recall parameters of experiment
n=500
k=5
theta_intra_min, theta_intra_max = 0.4, 0.6
theta_inter_min, theta_inter_max = 0.01, 0.2
theta = AssociativeFullyRandom(
                K=k,
                min_inside_classes=theta_intra_min,
                max_inside_classes=theta_intra_max,
                min_outside_classes=theta_inter_min,
                max_outside_classes=theta_inter_max,
            ).Theta
graphon_mat = StepGraphon(theta, 1/k)
# Set Adjacency
A = graphon_mat.draw(rho=1, n=n, exchangeable=False)

In [ ]:
from pygraphon.estimators.networkhistogram.HistogramEstimator import HistogramEstimator

my_hist = HistogramEstimator()
my_hist.fit(graph= A, bandwidthHist= None)


In [ ]:
my_hist.get_graphon()

In [ ]:
from SmoothNetHist import SmoothNetHist
smooth_test = SmoothNetHist(bandwidth=1/k)
test_return_first = smooth_test._first_fit_hist(A)

# Testing if the smoothing along the tensor are the same

In [ ]:
test_return_first.get_graphon()

In [ ]:
smoothing_new = smooth_test._smoothing_histLC(test_return_first, A)

In [ ]:
smoothing_old,_,_ = smoothing_histLC(test_return_first.get_graphon(), A)

In [ ]:
np.sum(smoothing_new- smoothing_old)

# Testing if the tensor of smoothed graphon returned is the same

In [ ]:
tensor_new = smooth_test._flat_to_tensor(smoothing_new, test_return_first.get_graphon(), A.shape[0])
nclust = smoothing_old.shape[0]
nrow = test_return_first.get_graphon().graphon.shape[0]
tensor_old = flat_to_tensor(smoothing_old, nclust, nrow, test_return_first.get_graphon(), n)
np.sum(tensor_new - tensor_old)